In [1]:
!nvidia-smi

Wed Apr 22 11:40:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
%%writefile vector_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda.h>

/* =========================
   CUDA KERNEL
   ========================= */
__global__ void vectorAdd(int *A, int *B, int *C, int N)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N)
        C[i] = A[i] + B[i];
}

/* =========================
   MAIN
   ========================= */
int main()
{
    int testSizes[4] = {10000, 50000, 100000, 500000};

    for (int t = 0; t < 4; t++)
    {
        int N = testSizes[t];
        printf("\n========== TEST CASE %d (N = %d) ==========\n", t + 1, N);

        size_t size = N * sizeof(int);

        int *h_A = (int*)malloc(size);
        int *h_B = (int*)malloc(size);
        int *h_C = (int*)malloc(size);
        int *h_C_seq = (int*)malloc(size);

        // initialize
        for (int i = 0; i < N; i++)
        {
            h_A[i] = i;
            h_B[i] = i * 2;
        }

        /* =========================
           SEQUENTIAL (CPU)
           ========================= */
        clock_t start_cpu = clock();

        for (int i = 0; i < N; i++)
            h_C_seq[i] = h_A[i] + h_B[i];

        clock_t end_cpu = clock();
        double cpu_time = (double)(end_cpu - start_cpu) / CLOCKS_PER_SEC;

        /* =========================
           CUDA (GPU)
           ========================= */
        int *d_A, *d_B, *d_C;

        cudaMalloc(&d_A, size);
        cudaMalloc(&d_B, size);
        cudaMalloc(&d_C, size);

        cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

        int threads = 256;
        int blocks = (N + threads - 1) / threads;

        cudaEvent_t start, stop;
        cudaEventCreate(&start);
        cudaEventCreate(&stop);

        cudaEventRecord(start);

        vectorAdd<<<blocks, threads>>>(d_A, d_B, d_C, N);
        cudaDeviceSynchronize();

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float gpu_time;
        cudaEventElapsedTime(&gpu_time, start, stop); // ms

        cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

        /* =========================
           PERFORMANCE
           ========================= */
        double gpu_time_sec = gpu_time / 1000.0;
        double speedup = cpu_time / gpu_time_sec;

        /* =========================
           OUTPUT
           ========================= */
        printf("CPU Time = %f sec\n", cpu_time);
        printf("GPU Time = %f ms\n", gpu_time);

        printf("Speedup = %f\n", speedup);

        printf("Sample Output: C[0] = %d, C[N-1] = %d\n",
               h_C[0], h_C[N - 1]);

        /* Cleanup */
        free(h_A);
        free(h_B);
        free(h_C);
        free(h_C_seq);

        cudaFree(d_A);
        cudaFree(d_B);
        cudaFree(d_C);
    }

    return 0;
}

Writing vector_add.cu


In [4]:
!nvcc vector_add.cu -o vec_add
!./vec_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

========== TEST CASE 1 (N = 10000) ==========
CPU Time = 0.000064 sec
GPU Time = 178.735001 ms
Speedup = 0.000358
Sample Output: C[0] = 0, C[N-1] = 29997

========== TEST CASE 2 (N = 50000) ==========
CPU Time = 0.000194 sec
GPU Time = 0.027936 ms
Speedup = 6.944444
Sample Output: C[0] = 0, C[N-1] = 149997

========== TEST CASE 3 (N = 100000) ==========
CPU Time = 0.000373 sec
GPU Time = 2.059616 ms
Speedup = 0.181102
Sample Output: C[0] = 0, C[N-1] = 299997

========== TEST CASE 4 (N = 500000) ==========
CPU Time = 0.002876 sec
GPU Time = 0.035968 ms
Speedup = 79.959968
Sample Output: C[0] = 0, C[N-1] = 1499997


In [5]:
%%writefile matrix_mul.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda.h>

/* =========================
   CUDA KERNEL
   ========================= */
__global__ void matMul(int *A, int *B, int *C, int N)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        int sum = 0;
        for (int k = 0; k < N; k++)
        {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

/* =========================
   MAIN
   ========================= */
int main()
{
    int testSizes[4] = {2, 10, 50, 200};  // increasing complexity

    for (int t = 0; t < 4; t++)
    {
        int N = testSizes[t];
        printf("\n========== TEST CASE %d (N = %d) ==========\n", t + 1, N);

        int size = N * N * sizeof(int);

        int *h_A = (int*)malloc(size);
        int *h_B = (int*)malloc(size);
        int *h_C = (int*)malloc(size);
        int *h_C_seq = (int*)malloc(size);

        // initialize
        for (int i = 0; i < N * N; i++)
        {
            h_A[i] = rand() % 10;
            h_B[i] = rand() % 10;
        }

        /* =========================
           SEQUENTIAL (CPU)
           ========================= */
        clock_t start_cpu = clock();

        for (int i = 0; i < N; i++)
        {
            for (int j = 0; j < N; j++)
            {
                int sum = 0;
                for (int k = 0; k < N; k++)
                {
                    sum += h_A[i * N + k] * h_B[k * N + j];
                }
                h_C_seq[i * N + j] = sum;
            }
        }

        clock_t end_cpu = clock();
        double cpu_time = (double)(end_cpu - start_cpu) / CLOCKS_PER_SEC;

        /* =========================
           CUDA (GPU)
           ========================= */
        int *d_A, *d_B, *d_C;

        cudaMalloc(&d_A, size);
        cudaMalloc(&d_B, size);
        cudaMalloc(&d_C, size);

        cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

        dim3 threads(16, 16);
        dim3 blocks((N + 15) / 16, (N + 15) / 16);

        cudaEvent_t start, stop;
        cudaEventCreate(&start);
        cudaEventCreate(&stop);

        cudaEventRecord(start);

        matMul<<<blocks, threads>>>(d_A, d_B, d_C, N);
        cudaDeviceSynchronize();

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float gpu_time;
        cudaEventElapsedTime(&gpu_time, start, stop); // ms

        cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

        /* =========================
           PERFORMANCE
           ========================= */
        double gpu_time_sec = gpu_time / 1000.0;
        double speedup = cpu_time / gpu_time_sec;

        /* =========================
           OUTPUT
           ========================= */
        printf("CPU Time = %f sec\n", cpu_time);
        printf("GPU Time = %f ms\n", gpu_time);
        printf("Speedup = %f\n", speedup);

        // print small matrix only
        if (N <= 10)
        {
            printf("\nResult Matrix:\n");
            for (int i = 0; i < N * N; i++)
            {
                printf("%d ", h_C[i]);
                if ((i + 1) % N == 0) printf("\n");
            }
        }

        /* Cleanup */
        free(h_A);
        free(h_B);
        free(h_C);
        free(h_C_seq);

        cudaFree(d_A);
        cudaFree(d_B);
        cudaFree(d_C);
    }

    return 0;
}

Writing matrix_mul.cu


In [6]:
!nvcc matrix_mul.cu -o mat_mul
!./mat_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

========== TEST CASE 1 (N = 2) ==========
CPU Time = 0.000001 sec
GPU Time = 24.842848 ms
Speedup = 0.000040

Result Matrix:
53 29 
48 27 

========== TEST CASE 2 (N = 10) ==========
CPU Time = 0.000008 sec
GPU Time = 0.020320 ms
Speedup = 0.393701

Result Matrix:
104 185 154 149 118 145 187 224 67 58 
268 265 282 299 215 235 227 406 286 171 
201 245 196 205 193 239 223 328 193 122 
219 228 207 188 179 224 180 346 204 134 
257 280 235 243 205 269 248 398 240 162 
150 157 197 135 181 184 170 271 109 81 
236 281 263 240 278 270 284 464 216 248 
222 253 214 225 164 206 221 363 181 176 
215 270 208 199 225 292 263 390 176 174 
251 309 271 292 266 281 309 452 254 210 

========== TEST CASE 3 (N = 50) ==========
CPU Time = 0.000364 sec
GPU Time = 0.019392 ms
Speedup = 18.770627

========== TEST CASE 4 (N = 

In [ ]:
%%writefile vector_add_vis.cu
#include <stdio.h>
#include <cuda.h>

#define N 12   // small so we can SEE output

__global__ void vectorAdd(int *A, int *B, int *C)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];

        // VISUALIZATION OF PARALLELISM
        printf("Thread %d (Block %d) computing C[%d] = %d + %d = %d\n",
               threadIdx.x, blockIdx.x, i, A[i], B[i], C[i]);
    }
}

int main()
{
    int *h_A, *h_B, *h_C;
    int *d_A, *d_B, *d_C;

    size_t size = N * sizeof(int);

    h_A = (int*)malloc(size);
    h_B = (int*)malloc(size);
    h_C = (int*)malloc(size);

    for (int i = 0; i < N; i++)
    {
        h_A[i] = i;
        h_B[i] = i * 2;
    }

    printf("\nInput 1 A[0..N-1]:\n");
    for (int i = 0; i < N; i++)
        printf("%d ", h_A[i]);

    printf("\nInput 2 B[0..N-1]:\n");
    for (int i = 0; i < N; i++)
        printf("%d ", h_B[i]);

    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 threads(8);
    dim3 blocks(4);

    vectorAdd<<<blocks, threads>>>(d_A, d_B, d_C);

    cudaDeviceSynchronize();

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    printf("\nFinal Output C[0..N-1]:\n");
    for (int i = 0; i < N; i++)
        printf("%d ", h_C[i]);

    printf("\n");

    return 0;
}

Overwriting vector_add_vis.cu


In [8]:
!nvcc vector_add_vis.cu -o vec_vis
!./vec_vis

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
cc1plus: fatal error: vector_add_vis.cu: No such file or directory
compilation terminated.
/bin/bash: line 1: ./vec_vis: No such file or directory


In [7]:
%%writefile matrix_mul_opt.cu
#include <stdio.h>
#include <cuda.h>

#define N 4

__global__ void matMul(int *A, int *B, int *C)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        int sum = 0;

        for (int k = 0; k < N; k++)
        {
            sum += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = sum;

        // VISUALIZE THREAD WORK
        printf("Thread(%d,%d) Block(%d,%d) computes C[%d][%d] = %d\n",
               threadIdx.x, threadIdx.y,
               blockIdx.x, blockIdx.y,
               row, col, sum);
    }
}

int main()
{
    int size = N * N * sizeof(int);

    int *h_A, *h_B, *h_C;
    int *d_A, *d_B, *d_C;

    h_A = (int*)malloc(size);
    h_B = (int*)malloc(size);
    h_C = (int*)malloc(size);

    for (int i = 0; i < N * N; i++)
    {
        h_A[i] = 1;
        h_B[i] = 2;
    }

    printf("\nFirst Matrix:\n");

    for (int i = 0; i < N * N; i++)
    {
        printf("%d ", h_A[i]);
        if ((i + 1) % N == 0) printf("\n");
    }

    printf("\nSecond Matrix:\n");

    for (int i = 0; i < N * N; i++)
    {
        printf("%d ", h_B[i]);
        if ((i + 1) % N == 0) printf("\n");
    }

    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 threads(2,2);
    dim3 blocks(2,2);

    matMul<<<blocks, threads>>>(d_A, d_B, d_C);

    cudaDeviceSynchronize();

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    printf("\nFinal Matrix:\n");

    for (int i = 0; i < N * N; i++)
    {
        printf("%d ", h_C[i]);
        if ((i + 1) % N == 0) printf("\n");
    }

    return 0;
}

Writing matrix_mul_opt.cu


In [9]:
!nvcc matrix_mul_opt.cu -o mat_opt
!./mat_opt

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

First Matrix:
1 1 1 1 
1 1 1 1 
1 1 1 1 
1 1 1 1 

Second Matrix:
2 2 2 2 
2 2 2 2 
2 2 2 2 
2 2 2 2 
Thread(0,0) Block(0,1) computes C[2][0] = 8
Thread(1,0) Block(0,1) computes C[2][1] = 8
Thread(0,1) Block(0,1) computes C[3][0] = 8
Thread(1,1) Block(0,1) computes C[3][1] = 8
Thread(0,0) Block(1,1) computes C[2][2] = 8
Thread(1,0) Block(1,1) computes C[2][3] = 8
Thread(0,1) Block(1,1) computes C[3][2] = 8
Thread(1,1) Block(1,1) computes C[3][3] = 8
Thread(0,0) Block(0,0) computes C[0][0] = 8
Thread(1,0) Block(0,0) computes C[0][1] = 8
Thread(0,1) Block(0,0) computes C[1][0] = 8
Thread(1,1) Block(0,0) computes C[1][1] = 8
Thread(0,0) Block(1,0) computes C[0][2] = 8
Thread(1,0) Block(1,0) computes C[0][3] = 8
Thread(0,1) Block(1,0) computes C[1][2] = 8
Thread(1,1) Block(1,0) computes C[1][3] = 8

Final